# IEEE Grids for Testing

This notebook demonstrates the **full synthesis pipeline** — from loading a real IEEE grid, extracting its topological parameters, generating a synthetic clone, and then assigning bus types, generation/load capacities, dispatch, and transmission-line parameters.

In [ ]:
# Install a fresh copy of powergrid_synth from GitHub.
import importlib.metadata
import os
import sys

was_already_imported = "powergrid_synth" in sys.modules

!pip uninstall -y powergrid_synth
!pip install -q --force-reinstall --no-cache-dir --no-deps \
    git+https://github.com/cookbook-ms/Chung_Lu_Chain-synthesizer.git@main
# Pin numpy & scipy to the versions already loaded in this runtime, then
# install the optional export and power-flow dependencies.
!pip install -q \
    "numpy==$(python3 -c 'import numpy; print(numpy.__version__)')" \
    "scipy==$(python3 -c 'import scipy; print(scipy.__version__)')" \
    pandapower lightsim2grid pypowsybl pypowsybl-jupyter

installed_version = importlib.metadata.version("powergrid_synth")
print(f"Installed powergrid_synth {installed_version}")

if was_already_imported:
    print("Restarting the runtime so the fresh package replaces the previously loaded module.")
    os.kill(os.getpid(), 9)

In [ ]:
import pandapower as pp
import pandapower.networks as pn
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

from powergrid_synth import (
    PowerGridGenerator,
    BusTypeAllocator,
    GraphComparator,
    GridVisualizer,
    CapacityAllocator,
    LoadAllocator,
    GenerationDispatcher,
    TransmissionLineAllocator,
    pandapower_to_nx,
    nx_to_pandapower,
    extract_topology_params_from_graph,
)

## Load an IEEE grid using pandapower

In [ ]:
print("\n[1] Loading Reference Grid (IEEE)...")
net_real = pn.case118()
graph_real = pandapower_to_nx(net_real)
base_kv_list = graph_real.graph['base_kv_map']
print(f"Loaded {graph_real.number_of_nodes()} nodes and {graph_real.number_of_edges()} edges.")

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
ax = pp.plotting.simple_plot(net_real, ax=ax)

## Generate a synthetic grid

### Extract Topology Characteristics from Graph

In [ ]:
print("\n[2] Analyzing Reference Topology...")
params = extract_topology_params_from_graph(graph_real)

### PowerGridGenerator

In [ ]:
print("\n[3] Generating Synthetic Clone...")
gen = PowerGridGenerator(seed=53)
synthetic_graph = gen.generate_grid(
    degrees_by_level=params['degrees_by_level'],
    diameters_by_level=params['diameters_by_level'],
    transformer_degrees=params['transformer_degrees'],
    keep_lcc=True
)

## Analysis

### Compare some metrics globally

In [ ]:
print("\n[5] Running Comparative Analysis...")
comparator = GraphComparator(
    synth_graph=synthetic_graph, 
    ref_graph=graph_real, 
    synth_label="Synthetic", 
    ref_label="IEEE grid"
)

In [ ]:
comparator.print_metric_comparison(title="GLOBAL TOPOLOGY COMPARISON")

### Plot the global node degree distribution for two grids

In [ ]:
comparator.plot_degree_comparison(log_scale=True, fig_size=(6,4), show_lines=True)

### Compare the histograms of node degrees for each voltage level

In [ ]:
comparator.plot_all_levels_comparison(False)

### Compare other topology metrics per voltage level

In [ ]:
comparator.plot_level_topology_comparison()

## Visualizations

In [ ]:
viz = GridVisualizer()

viz.plot_grid(
    graph_real, 
    layout='kamada_kawai',
    title="IEEE grid",
    figsize=(6, 4)
)

In [ ]:
viz.plot_grid(
    synthetic_graph, 
    layout='kamada_kawai',
    title="Synthetic Grid",
    figsize=(6, 4)
)

## Bus type assignment

In [ ]:
print("\n[4] Allocating Bus Types via AIS...")
allocator = BusTypeAllocator(synthetic_graph, entropy_model=0, bus_type_ratio=[80,60,0])
bus_types = allocator.allocate(max_iter=50)
nx.set_node_attributes(synthetic_graph, bus_types, name="bus_type")

In [ ]:
counts = Counter(bus_types.values())
total = sum(counts.values())
print(f"-----> Assignment Complete:")
print(f"       Generators: {counts['Gen']} ({counts['Gen']/total:.1%})")
print(f"       Loads:      {counts['Load']} ({counts['Load']/total:.1%})")
print(f"       Connectors: {counts['Conn']} ({counts['Conn']/total:.1%})")

print("\n[5] Visualizing Bus Types & Edge Styles...")
viz.plot_bus_types(
    synthetic_graph, 
    layout='kamada_kawai', 
    title="Bus Types & Transmission Lines", 
    figsize=(7,5)
)

## Generation capacities and load settings

In [ ]:
print("\n[6] Allocating Capacity...")
cap_allocator = CapacityAllocator(synthetic_graph)
capacities = cap_allocator.allocate()
total_cap = sum(capacities.values())
print(f"Total Generation: {total_cap:.2f} MW")
nx.set_node_attributes(synthetic_graph, capacities, name="pg_max")

In [ ]:
sorted_gens = sorted(capacities.items(), key=lambda x: x[1], reverse=True)
print("\nTop 5 Generators by Capacity:")
for node, cap in sorted_gens[:5]:
    print(f"  Node {node}: {cap:.2f} MW (Degree: {synthetic_graph.degree(node)})")

In [ ]:
print("\n[7] Allocating Loads ...")
load_allocator = LoadAllocator(synthetic_graph, ref_sys_id=1)
loads = load_allocator.allocate(loading_level='H')
nx.set_node_attributes(synthetic_graph, loads, name="pl")

total_load = sum(loads.values())
print(f"Total Load: {total_load:.2f} MW")
print(f"System Loading: {total_load/total_cap:.1%}")

In [ ]:
load_vals = list(loads.values())
caps = list(capacities.values())

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(caps, bins=30, color='skyblue', edgecolor='black')
plt.title("Generator Capacity Distribution")
plt.xlabel("Capacity (MW)")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(load_vals, bins=30, color='orange', edgecolor='black')
plt.title("Load Size Distribution")
plt.xlabel("Load (MW)")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Generation dispatch and transmission-line allocation

In [ ]:
print("\n[8] Dispatching Generation...")
dispatcher = GenerationDispatcher(synthetic_graph, ref_sys_id=1)
dispatch = dispatcher.dispatch()
nx.set_node_attributes(synthetic_graph, dispatch, name="pg")
total_gen = sum(dispatch.values())
print(f"-> Total Power Dispatched: {total_gen:.2f} MW")
print(f"-> Generation Reserve: {total_cap - total_gen:.2f} MW")

In [ ]:
print("\n[9] Allocating Transmission Lines (Impedance & Capacity)...")
trans_allocator = TransmissionLineAllocator(synthetic_graph, ref_sys_id=1)
line_caps = trans_allocator.allocate()

total_lines = len(line_caps)
avg_cap = sum(line_caps.values()) / total_lines if total_lines > 0 else 0
print(f"-> Allocated {total_lines} Lines")
print(f"-> Average Line Capacity: {avg_cap:.2f} MVA")

In [ ]:
print("-> Plotting Generation vs Load Bubbles...")
viz.plot_load_gen_bubbles(synthetic_graph, layout='kamada_kawai', title=f"Generation vs Load (Total: {total_load:.0f} MW)")

In [ ]:
viz.plot_impedance(synthetic_graph, layout='kamada_kawai', title="Grid Impedance Map (Blue=Low Z, Red=High Z)")

## Compatible with pandapower

[pandapower](https://www.pandapower.org/) provides **Newton-Raphson AC** (`pp.runpp`) and **linear DC** (`pp.rundcpp`) power-flow solvers, and export to **JSON, Excel, SQLite, Pickle**.

> **Note:** `pp.runpp` (AC) may not converge for large synthetic grids; `pp.rundcpp` (DC) always works. For AC power flow on large grids, use pypowsybl's `run_ac` solver instead (shown below).

In [ ]:
synthetic_net = nx_to_pandapower(synthetic_graph, base_kv_map=base_kv_list)
synthetic_net

In [ ]:
pp.rundcpp(synthetic_net)
synthetic_net

#### Underlying pandapower built-in export (JSON example)

In [ ]:
pp.to_json(synthetic_net, filename='ieee118_syn.json')

## Compatible with PowSyBl

[pypowsybl](https://pypowsybl.readthedocs.io/) provides **AC and DC load-flow solvers** (`run_ac`, `run_dc`), interactive grid visualisation, and export to **CGMES, XIIDM, MATPOWER, PSS/E, UCTE, AMPL, BIIDM, JIIDM**.

#### Convert from pandapower to pypowsybl

In [ ]:
import pypowsybl as ppl
from powergrid_synth import pandapower_to_pypowsybl

ppl_net = pandapower_to_pypowsybl(synthetic_net)
ppl_net

#### Run AC PF solver using PowSyBl

In [ ]:
ppl.loadflow.run_ac(ppl_net)

#### Interactive grid visualizer from PowSyBl

In [ ]:
from pypowsybl_jupyter import network_explorer, nad_explorer, display_nad

In [ ]:
nad_explorer(ppl_net, depth=3)

In [ ]:
ppl_net.get_network_area_diagram()

### Supported data export formats

PowerGridSynth provides a unified `GridExporter` class that wraps the built-in export functions of **pandapower** and **pypowsybl**, supporting **12+ industry-standard data formats** out of the box.

| Via | Formats | Methods |
|-----|---------|---------|
| **pandapower** | JSON, Excel, SQLite, Pickle | `to_json()`, `to_excel()`, `to_sqlite()`, `to_pickle()` |
| **pypowsybl** | CGMES, XIIDM, MATPOWER, PSS/E, UCTE, AMPL, BIIDM, JIIDM | `to_cgmes()`, `to_matpower()`, `to_psse()`, `to_pypowsybl(format=...)` |

In [ ]:
from powergrid_synth import GridExporter

exporter = GridExporter(synthetic_graph, base_kv_map=base_kv_list)

# --- pandapower-based exports ---
exporter.to_json("output/ieee118_syn.json")
# exporter.to_excel("output/ieee118_syn.xlsx")
# exporter.to_sqlite("output/ieee118_syn.sqlite")
# exporter.to_pickle("output/ieee118_syn.p")

# --- pypowsybl-based exports ---
exporter.to_cgmes("output/ieee118_syn_cgmes")
exporter.to_matpower("output/ieee118_syn")
exporter.to_psse("output/ieee118_syn")
exporter.to_pypowsybl("output/ieee118_syn.xiidm", format="XIIDM")

#### Underlying pypowsybl built-in export formats

In [ ]:
ppl.network.get_export_formats()

In [ ]:
ppl_net.save('ieee118_syn.cgmes')
# or ppl_net.save('ieee118_syn', format='CGMES')